# Economía del Comportamiento (Behavioral Economics)

Notebook reconstruida con `numpy`, `pandas`, `scipy`, `sympy`, `seaborn`, `matplotlib` y `statsmodels` para que la parte computacional sea más fiel a cómo se trabaja de verdad en investigación aplicada.

**Objetivos**
- Comparar utilidad esperada y teoría prospectiva.
- Visualizar curvas de valor y ponderación de probabilidades.
- Mostrar reversión de preferencias bajo descuento cuasi-hiperbólico.
- Usar simulación + `statsmodels` para estimar un efecto de default.


## Tabla de Contenidos

1. Utilidad esperada como benchmark.
2. Teoría prospectiva y curvas conductuales.
3. Mapa de elección entre lotería segura y riesgosa.
4. Animación del present bias.
5. Preferencias sociales y calor de aceptación en ultimátum.
6. Simulación y estimación de un efecto default con `statsmodels`.
7. Ejercicios.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import scipy.stats as st
import sympy as sp
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.optimize import brentq
from scipy.special import expit
import statsmodels.api as sm

sns.set_theme(style="whitegrid", context="talk")
rng = np.random.default_rng(20260316)


def display_animation(anim):
    html = HTML(anim.to_jshtml(default_mode="loop"))
    plt.close(anim._fig)
    display(html)


def prospect_value(x, alpha=0.88, beta=0.88, loss_aversion=2.25):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x, dtype=float)
    mask = x >= 0
    out[mask] = np.power(x[mask], alpha)
    out[~mask] = -loss_aversion * np.power(-x[~mask], beta)
    return out


def prelec_weight(p, gamma=0.61):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.exp(-np.power(-np.log(p), gamma))


def expected_utility(lottery, utility):
    return sum(prob * utility(value) for prob, value in lottery)


def certainty_equivalent(lottery, utility, bracket):
    target = expected_utility(lottery, utility)
    return brentq(lambda c: utility(c) - target, *bracket)


def beta_delta_value(amount, payoff_date, eval_date, beta=0.7, delta=0.95):
    if payoff_date < eval_date:
        return np.nan
    delay = payoff_date - eval_date
    return amount if delay == 0 else beta * (delta ** delay) * amount


def fehr_schmidt(x_self, x_other, alpha=1.4, beta=0.4):
    envy = np.maximum(x_other - x_self, 0)
    guilt = np.maximum(x_self - x_other, 0)
    return x_self - alpha * envy - beta * guilt


x_sym, alpha_sym, beta_sym, lambda_sym = sp.symbols("x alpha beta lambda", positive=True)
value_piecewise = sp.Piecewise((x_sym**alpha_sym, x_sym >= 0), (-lambda_sym * (-x_sym) ** beta_sym, True))
display(sp.Eq(sp.Symbol("v(x)"), value_piecewise))


## 1. Benchmark: utilidad esperada y paradoja de Allais

La utilidad esperada sigue siendo el benchmark normativo. La pregunta central de la economía del comportamiento no es si “sirve o no”, sino **cuándo y cómo falla empíricamente**.

Usaremos una especificación CRRA para comparar loterías del problema de Allais, y luego contrastaremos ese benchmark con mecanismos conductuales.


In [ ]:
initial_wealth = 2_000_000
gamma = 1.2


def u_crra(w):
    if np.isclose(gamma, 1.0):
        return np.log(w)
    return (w ** (1 - gamma) - 1) / (1 - gamma)


lotteries = {
    "A seguro": [(1.00, initial_wealth + 1_000_000)],
    "B riesgoso": [(0.89, initial_wealth + 1_000_000), (0.10, initial_wealth + 5_000_000), (0.01, initial_wealth)],
    "C": [(0.11, initial_wealth + 1_000_000), (0.89, initial_wealth)],
    "D": [(0.10, initial_wealth + 5_000_000), (0.90, initial_wealth)],
}

rows = []
for name, lot in lotteries.items():
    ev = sum(p * x for p, x in lot)
    eu = expected_utility(lot, u_crra)
    ce = certainty_equivalent(lot, u_crra, (initial_wealth, initial_wealth + 5_000_000))
    rows.append((name, ev, eu, ce))

allais_df = pd.DataFrame(rows, columns=["loteria", "valor_esperado", "utilidad_esperada", "equivalente_cierto"])
print(allais_df.round(2))

fig, ax = plt.subplots(figsize=(9, 5), facecolor="#fbfaf7")
sns.barplot(data=allais_df, x="loteria", y="equivalente_cierto", hue="loteria", palette="crest", legend=False, ax=ax)
ax.set_title("Equivalente cierto bajo utilidad esperada CRRA")
ax.set_ylabel("equivalente cierto")
plt.show()


## 2. Teoría prospectiva: función de valor y ponderación de probabilidades

Kahneman y Tversky propusieron dos ingredientes centrales:

- curvatura distinta en ganancias y pérdidas;
- transformación no lineal de probabilidades.

Estas piezas se prestan muy bien a visualización, y ahí `seaborn` y `matplotlib` realmente mejoran la explicabilidad.


In [ ]:
x_grid = np.linspace(-200, 200, 600)
p_grid = np.linspace(0.001, 0.999, 400)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), facecolor="#fbfaf7")
sns.lineplot(x=x_grid, y=prospect_value(x_grid), ax=axes[0], color="#c44e52", lw=3)
axes[0].axvline(0, color="black", lw=1)
axes[0].axhline(0, color="black", lw=1)
axes[0].set_title("Función de valor prospectiva")
axes[0].set_xlabel("resultado relativo al punto de referencia")
axes[0].set_ylabel("valor psicológico")

sns.lineplot(x=p_grid, y=prelec_weight(p_grid), ax=axes[1], color="#4c72b0", lw=3, label="w(p)")
sns.lineplot(x=p_grid, y=p_grid, ax=axes[1], color="gray", ls="--", label="línea 45°")
axes[1].set_title("Ponderación de probabilidades (Prelec)")
axes[1].set_xlabel("probabilidad objetiva")
axes[1].set_ylabel("peso decisional")
axes[1].legend()

fig.tight_layout()
plt.show()


## 3. Mapa de elección: opción segura vs. lotería riesgosa

Tomamos una opción segura de 100 y la comparamos con una lotería de la forma:
\[
(p, x_{\text{alto}};\ 1-p, 0).
\]

El mapa siguiente muestra dónde la teoría prospectiva favorece la opción segura aun cuando el valor esperado monetario pueda ser mayor en la lotería.


In [ ]:
probs = np.linspace(0.05, 0.95, 40)
highs = np.linspace(100, 240, 50)
safe_gain = 100

heat = np.zeros((len(highs), len(probs)))
for i, high in enumerate(highs):
    for j, p in enumerate(probs):
        risky_score = prelec_weight(p) * prospect_value(high) + (1 - prelec_weight(p)) * prospect_value(0)
        safe_score = prospect_value(safe_gain)
        heat[i, j] = risky_score - safe_score

fig, ax = plt.subplots(figsize=(11, 6), facecolor="#fbfaf7")
sns.heatmap(
    heat,
    cmap="coolwarm",
    center=0,
    xticklabels=np.round(probs, 2),
    yticklabels=np.round(highs, 0).astype(int),
    ax=ax,
)
ax.set_title("Riesgosa menos segura bajo teoría prospectiva")
ax.set_xlabel("probabilidad del premio alto")
ax.set_ylabel("premio alto")
plt.show()


## 4. Animación: reversión de preferencias bajo present bias

Ahora fijamos dos pagos:

- Opción A: 100 en la fecha 5.
- Opción B: 120 en la fecha 6.

Bajo descuento cuasi-hiperbólico, la evaluación cambia cuando la fecha presente se acerca a los pagos. La animación muestra exactamente esa reversión.


In [ ]:
eval_dates = np.arange(0, 6)
values_a = np.array([beta_delta_value(100, 5, t, beta=0.7, delta=0.95) for t in eval_dates])
values_b = np.array([beta_delta_value(120, 6, t, beta=0.7, delta=0.95) for t in eval_dates])

fig, ax = plt.subplots(figsize=(9, 5), facecolor="#fbfaf7")
ax.set_ylim(0, max(values_b) * 1.2)
bars = ax.bar(["100 en t=5", "120 en t=6"], [values_a[0], values_b[0]], color=["#4c72b0", "#dd8452"])
title = ax.set_title("Perspectiva desde t = 0")
footer = ax.text(0.5, -0.18, "", transform=ax.transAxes, ha="center", fontsize=12)

def update(frame):
    bars[0].set_height(values_a[frame])
    bars[1].set_height(values_b[frame])
    choice = "A" if values_a[frame] >= values_b[frame] else "B"
    title.set_text(f"Perspectiva desde t = {eval_dates[frame]}")
    footer.set_text(f"Se elige la opción {choice}")
    return (*bars, title, footer)

anim = FuncAnimation(fig, update, frames=len(eval_dates), interval=700, blit=True)
display_animation(anim)


## 5. Preferencias sociales: utilidad del respondedor en ultimátum

Usaremos una parametrización de Fehr-Schmidt y construiremos un mapa de calor donde varía:

- la oferta al respondedor;
- el parámetro de aversión a la desigualdad desfavorable.

Así se ve mejor cuándo una oferta deja de ser “aceptable” incluso si es monetariamente positiva.


In [ ]:
offers = np.arange(1, 10)
alpha_grid = np.linspace(0.2, 2.4, 60)
utility_map = np.zeros((len(alpha_grid), len(offers)))

for i, alpha_envy in enumerate(alpha_grid):
    for j, offer in enumerate(offers):
        utility_map[i, j] = fehr_schmidt(offer, 10 - offer, alpha=alpha_envy, beta=0.3)

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#fbfaf7")
sns.heatmap(
    utility_map,
    cmap="vlag",
    center=0,
    xticklabels=offers,
    yticklabels=np.round(alpha_grid[::5], 2),
    ax=ax,
)
ax.set_title("Utilidad del respondedor en ultimátum")
ax.set_xlabel("oferta al respondedor")
ax.set_ylabel("aversión a desigualdad desfavorable")
plt.show()


## 6. Defaults y estimación empírica con `statsmodels`

Simulamos una base de decisiones donde la elección depende de:

- la ventaja intrínseca del plan;
- el default;
- la interacción entre ambas.

Luego ajustamos un Logit para recuperar el efecto del default y graficamos probabilidades predichas. Esto hace la notebook mucho más cercana a una práctica de investigación aplicada.


In [ ]:
n = 1600
sim_df = pd.DataFrame(
    {
        "utility_gap": rng.normal(loc=0.0, scale=0.9, size=n),
        "default": rng.integers(0, 2, size=n),
    }
)
latent = -0.3 + 1.6 * sim_df["utility_gap"] + 1.1 * sim_df["default"] + 0.5 * sim_df["utility_gap"] * sim_df["default"]
sim_df["choice"] = rng.binomial(1, expit(latent))

X = sm.add_constant(
    pd.DataFrame(
        {
            "utility_gap": sim_df["utility_gap"],
            "default": sim_df["default"],
            "interaction": sim_df["utility_gap"] * sim_df["default"],
        }
    )
)
logit_res = sm.Logit(sim_df["choice"], X).fit(disp=False)

grid = np.linspace(sim_df["utility_gap"].min(), sim_df["utility_gap"].max(), 100)
pred_df = pd.DataFrame(
    {
        "utility_gap": np.r_[grid, grid],
        "default": np.r_[np.zeros_like(grid), np.ones_like(grid)],
    }
)
pred_df["interaction"] = pred_df["utility_gap"] * pred_df["default"]
pred_X = sm.add_constant(pred_df, has_constant="add")
pred_df["pred_prob"] = logit_res.predict(pred_X)

coef_table = pd.DataFrame(
    {
        "coef": logit_res.params,
        "std_err": logit_res.bse,
        "p_value": logit_res.pvalues,
    }
).round(4)
print(coef_table)

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#fbfaf7")
sns.lineplot(data=pred_df, x="utility_gap", y="pred_prob", hue="default", palette="Set2", lw=3, ax=ax)
ax.set_title("Probabilidad de elección según utilidad y default")
ax.set_ylabel("probabilidad predicha")
plt.show()


## 7. Ejercicios

1. Cambia el parámetro de pérdida en la función prospectiva y observa cómo cambia el mapa de elección.
2. Ajusta la animación de present bias para comparar dos pares de recompensas.
3. Reestima el Logit sin interacción. ¿Qué interpretación pierdes?
4. Busca el umbral mínimo de aceptación en el heatmap de ultimátum para distintos valores de $\alpha$.


In [ ]:
alt_scores = prospect_value(np.array([-100, -50, 50, 100]), loss_aversion=3.0)
print("Valores prospectivos alternativos:", alt_scores.round(2))

simple_X = sm.add_constant(sim_df[["utility_gap", "default"]])
simple_logit = sm.Logit(sim_df["choice"], simple_X).fit(disp=False)
print("AIC modelo con interacción:", round(logit_res.aic, 3))
print("AIC modelo sin interacción:", round(simple_logit.aic, 3))


## Referencias

- Kahneman, D. y Tversky, A. (1979). *Prospect Theory*.
- Laibson, D. (1997). *Golden Eggs and Hyperbolic Discounting*.
- Fehr, E. y Schmidt, K. (1999). *A Theory of Fairness, Competition, and Cooperation*.
- DellaVigna, S. (2009). *Psychology and Economics*.
